# Stress test 3 — deduplication behaviour & surprises

`dedupe=True` merges content-identical nodes. Powerful for idempotent re-runs, but it has surprises worth understanding.

In [1]:
import shutil
import os
import time
import ancestree
from ancestree import LineageStore

SCRATCH = "_stress"  # all stores written here; safe to delete
shutil.rmtree(SCRATCH, ignore_errors=True)
os.makedirs(SCRATCH, exist_ok=True)


def store(name, **kw):
    return LineageStore(f"{SCRATCH}/{name}", **kw)


def node_dirs(s):
    return [d for d in os.listdir(s.root) if not d.startswith(".")]


print("ancestree", getattr(ancestree, "__version__", "?"))

ancestree 0.1.0


## ⚠️ Finding #4 — identical re-runs silently merge (distinct events are lost)

In [2]:
s = store("runs", dedupe=True, chunk=False)


def run_experiment(tag):
    with s.create_node(step_type="run") as n:
        (n / "out.csv").write_text("result\n1\n")  # identical output each run
        n.add_meta("pipeline", "v1")
    return n


a = run_experiment("monday")
time.sleep(0.01)
b = run_experiment("tuesday")  # same content a day later
print("two runs, same node_id?", a.node_id == b.node_id)
print("run nodes on disk:", len(node_dirs(s)), "(expected 2 distinct events, got 1)")
print(
    "Tuesday's run reports Monday's timestamp:",
    s.get_node(b.node_id).metadata["timestamp"]["value"],
)

two runs, same node_id? True
run nodes on disk: 1 (expected 2 distinct events, got 1)
Tuesday's run reports Monday's timestamp: 2026-06-25T14:29:47.523209+00:00


If you need to **count or audit every run**, dedupe will undercount — the second `with ... as node` hands you the *original* node, including its original timestamp and provenance. Add a distinguishing metadata value (e.g. a run id or timestamp) if every run must stay distinct.

In [3]:
# Make runs distinct by adding a unique field -> dedupe no longer merges
s2 = store("runs_distinct", dedupe=True)


def run_unique(i):
    with s2.create_node(step_type="run") as n:
        (n / "out.csv").write_text("result\n1\n")
        n.add_meta("run_id", i)  # unique per run
    return n


ids = {run_unique(i).node_id for i in range(3)}
print("distinct run nodes:", len(ids))

distinct run nodes: 3


## Provenance / time are excluded from identity (by design)

In [4]:
# Same content produced under different git commits / times still merges.
s3 = store("prov", dedupe=True)
with s3.create_node(step_type="x") as a:
    a.add_meta("k", 1)
with s3.create_node(step_type="x") as b:
    b.add_meta("k", 1)
print("merged across (hypothetically) different env/commit:", a.node_id == b.node_id)

merged across (hypothetically) different env/commit: True


## Correct distinctness: same content, different parent stays separate

In [5]:
s4 = store("branch", dedupe=True)
with s4.create_node(step_type="ingest") as r1:
    (r1 / "a").write_text("A")
with s4.create_node(step_type="ingest") as r2:
    (r2 / "b").write_text("B")
with s4.create_node(step_type="clean", parent=r1) as c1:
    (c1 / "c").write_text("SAME")
with s4.create_node(step_type="clean", parent=r2) as c2:
    (c2 / "c").write_text("SAME")
print("same content, different parent -> distinct:", c1.node_id != c2.node_id)

same content, different parent -> distinct: True


## ✅ Finding #5 (FIXED) — a parent from another store (or a bad id) is rejected

In [6]:
A = store("storeA")
B = store("storeB")
with A.create_node(step_type="x") as a:
    a.add_meta("k", 1)
try:
    with B.create_node(step_type="x", parent=a) as b:  # parent belongs to store A!
        b.add_meta("k", 1)
except ValueError as e:
    print("rejected at create_node:", e)

# An unknown id is rejected the same clear way (was: misleading error / silent root)
try:
    with B.create_node(step_type="x", parent="deadbeef"):
        pass
except ValueError as e:
    print("unknown id:", e)

rejected at create_node: Parent node 'b14b616a' is not present in this store. A parent must be a node already created in or loaded from this store; pass parent=None to create a root node.
unknown id: Parent node 'deadbeef' is not present in this store. A parent must be a node already created in or loaded from this store; pass parent=None to create a root node.
